# Global country scatter: growth rate vs required geological resource (Part 3 continuation)

A geographically resolved scatter in the style of Zhang et al. (NC,
2024) Fig 2, extended with the after-screening geological story. There
is one square scatter panel per V8 country/region, with x = sampled
growth rate `g` (%), y = required geological resource `C_sampled` (Gt,
log scale), and point colour = growth model (Logistic / Gompertz). The
technical/free MC background comes from `minimum`, `reference`,
`maximum`, and `growth10`. Three geological-constraint lines (NC-Future
style) mark the `minimum`, `reference`, and `maximum` scenario
`C_scenario` values (rounded). Screening-informed reconstruction markers
(this study, Part 3) plot, for every Part-2 failed scenario/model, the
Part-3 reconstruction resource `C_recon` (Gt), averaged across ascending
and descending orderings for visual compactness, with colour = scenario
and shape = model.

Inputs:
- Part 1 MC pools: `01_growth_model/output/v8_2026-06-01/samples_*.csv`
- Part 1 scenario scalars: `.../scalars_central.csv`
- Part 3 reconstruction: `03_feasible_growth/output/smooth_reconstruction_summary.csv`

This notebook lives in Part 3 (feasible-growth reconstruction) because it
compares the Part-3 geology-feasible reconstruction outputs against the
Part-1 scenario goals. Countries that pass all Part-2 deterministic screening
cases show only the MC background and goal lines, with no reconstruction marker.

Outputs (under `03_feasible_growth/output/`):
- `global_country_scatter/country_mc_scatter_points.csv`
- `global_country_scatter/country_reconstruction_points.csv`
- `figures/fig_global_country_growth_vs_resource.{png,pdf}`

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", '/private/tmp/mplconfig_iman2026')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

# Locate repo root
ROOT_CANDIDATES = [
    Path.cwd().resolve(), Path.cwd().resolve().parent,
    Path.cwd().resolve().parent.parent,
    Path('/Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026'),
]
for cand in ROOT_CANDIDATES:
    if (cand / '01_growth_model').exists():
        ROOT = cand; break
else:
    raise FileNotFoundError('repo root with 01_growth_model/ not found')

P1   = ROOT / '01_growth_model' / 'output' / 'v8_2026-06-01'   # MC samples + scenario scalars (v8)
P3   = ROOT / '03_feasible_growth' / 'output'                  # reconstruction + this figure
FIG  = P3 / 'figures'
TBL  = P3 / 'global_country_scatter'
FIG.mkdir(parents=True, exist_ok=True)
TBL.mkdir(parents=True, exist_ok=True)

# Config
# v8 study pool (all 10 regions have matched basins). The 2 that pass screening
# (Indonesia, Middle East) show MC + constraint lines but no reconstruction marker.
COUNTRIES = ['UK', 'US', 'EU', 'China', 'Middle East',
             'Australia', 'Canada', 'Indonesia', 'Thailand', 'Brazil']
# Free-MC technical scenarios that define the geological-constraint range.
SCATTER_SCENARIOS = ['minimum', 'reference', 'maximum', 'growth10']
CONSTRAINT_SCENARIOS = ['minimum', 'reference', 'maximum']

MODEL_COLOR    = {'Logistic': '#4C72B0', 'Gompertz': '#DD8452'}
MODEL_MARKER   = {'Logistic': 'o', 'Gompertz': '^'}
# Reconstruction lines: colour = scenario, line-style = model
MODEL_LINESTYLE = {'Logistic': '-', 'Gompertz': (0, (5, 2))}
SCENARIO_COLOR = {
    # Paul Tol "Muted" palette, colourblind-safe (Tol 2018).
    'minimum':   '#888888',  # medium grey (neutral baseline)
    'growth10':  '#DDCC77',  # sand
    'ipcc_low':  '#88CCEE',  # light blue
    'policy':    '#44AA99',  # teal
    'us1gt':     '#117733',  # green
    'reference': '#332288',  # indigo
    'ipcc_high': '#AA4499',  # purple
    'maximum':   '#CC6677',  # rose
}
SCENARIO_LABEL = {
    'reference': 'Reference', 'minimum': 'Minimum', 'maximum': 'Maximum',
    'growth10': 'Growth 10%', 'us1gt': 'US 1 Gt', 'policy': 'Policy',
    'ipcc_high': 'IPCC High', 'ipcc_low': 'IPCC Low',
}

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})
ROOT, P1, P3

In [ ]:
# 1. Assemble the MC scatter cloud per country
rows = []
for scen in SCATTER_SCENARIOS:
    path = P1 / f'samples_{scen}.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path)
    sub = df[df['Country'].isin(COUNTRIES)].copy()
    sub['scenario'] = scen
    sub['growth_rate_pct'] = sub['g'] * 100.0
    sub['resource_required_gt'] = sub['C_sampled']
    rows.append(sub[['Country', 'Model', 'scenario',
                      'growth_rate_pct', 'resource_required_gt']])
scatter = pd.concat(rows, ignore_index=True)
TBL.mkdir(parents=True, exist_ok=True)
scatter.to_csv(TBL / 'country_mc_scatter_points.csv', index=False)
GROWTH_MAX = float(scatter['growth_rate_pct'].max())
print(f'Scatter points: {len(scatter):,} | max growth rate = {GROWTH_MAX:.1f}%')

sc = pd.read_csv(P1 / 'scalars_central.csv')

# 2. Geological-constraint reference levels (min/ref/max C)
constraint_C = {}
for c in COUNTRIES:
    constraint_C[c] = {}
    for scen in CONSTRAINT_SCENARIOS:
        r = sc[(sc['Country'] == c) & (sc['Scenario'] == scen)]
        if not r.empty:
            constraint_C[c][scen] = round(float(r['C_scenario'].iloc[0]))

# 3. Goal vs feasible per failed (scenario, model, ORDERING).
#    Keep ascending AND descending (consistent with Fig P3-3a).
recon = pd.read_csv(P3 / 'smooth_reconstruction_summary.csv')
recon = recon[~recon['landmark_infeasible']].copy()
recon['C_recon_gt'] = recon['C_recon_order_gt'].round()
recon[['country','scenario','model','ordering','C_recon_gt']].to_csv(
    TBL / 'country_reconstruction_points.csv', index=False)

failed_scenarios = sorted(recon['scenario'].unique())

# goals[country][scenario] = scenario resource C (set goal)
goals = {c: {} for c in COUNTRIES}
for c in COUNTRIES:
    for scen in failed_scenarios:
        r = sc[(sc['Country'] == c) & (sc['Scenario'] == scen)]
        if not r.empty:
            goals[c][scen] = round(float(r['C_scenario'].iloc[0]))

# feasibles[country][scenario][model][ordering] = reconstruction C
feasibles = {c: {} for c in COUNTRIES}
for _, r in recon.iterrows():
    (feasibles.setdefault(r['country'], {})
              .setdefault(r['scenario'], {})
              .setdefault(r['model'], {}))[r['ordering']] = int(r['C_recon_gt'])

SCEN_ORDER = ['reference', 'growth10', 'policy', 'us1gt',
              'ipcc_low', 'ipcc_high', 'maximum']
def country_failed_scens(c):
    present = set(feasibles.get(c, {}).keys())
    return [s for s in SCEN_ORDER if s in present]

print(f'\nFailed scenarios: {failed_scenarios}')
n_pts = recon.groupby(['country']).size()
print('Reconstruction points per country (scenario×model×ordering):')
print(n_pts.to_string())

In [ ]:
# 4. Plot: per country = scatter (0 to 25%) + goal-to-feasible strip.
# Decluttered: the min/ref/max gridlines ARE the scenario goals, so no
# per-scenario goal squares; each scenario hangs its feasible markers
# below its goal gridline via a thin connector. Strip widened, markers
# shrunk.
from matplotlib.lines import Line2D

n = len(COUNTRIES)
ncols = 5            # 5 x 2 layout for the 10 v8 regions
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 5.0 * nrows))
axes = np.atleast_1d(axes).ravel()

X_DATA_MAX = 25.5
X_SEP      = 26.3
RAIL_X0, RAIL_X1 = 27.2, 37.5      # wider strip
XLIM       = 38.5
BOX_ASPECT = 0.74                  # slightly landscape, room for strip

FEAS_MARKER = {'Logistic': 'o', 'Gompertz': '^'}
panel_tags  = list('abcdefghijkl')

# which gridline a scenario's goal sits on (for the goal tick label)
def goal_level(country, scen):
    return goals[country].get(scen)

for idx, country in enumerate(COUNTRIES):
    ax = axes[idx]
    sub_c = scatter[scatter['Country'] == country]

    # scatter cloud (data region)
    for mdl in ('Logistic', 'Gompertz'):
        sm = sub_c[sub_c['Model'] == mdl]
        if sm.empty:
            continue
        ax.scatter(sm['growth_rate_pct'], sm['resource_required_gt'],
                    s=6, marker=MODEL_MARKER[mdl],
                    facecolors=MODEL_COLOR[mdl], edgecolors='none',
                    alpha=0.07, zorder=2, rasterized=True)

    # min/ref/max C gridlines = the scenario GOALS (full width)
    for scen in CONSTRAINT_SCENARIOS:
        if scen not in constraint_C[country]:
            continue
        cval = constraint_C[country][scen]
        ax.axhline(cval, color='#b5b5b5', linestyle=(0, (5, 3)),
                    linewidth=0.9, alpha=0.85, zorder=1)
        ax.text(0.25, cval, f'{scen} C={cval:,} (goal)',
                 ha='left', va='bottom', fontsize=6.8, color='#555555',
                 fontweight='bold',
                 zorder=3)

    ax.axvline(X_SEP, color='#dddddd', linewidth=0.8, zorder=1)

    # goal to feasible: connector from goal gridline down to feasibles
    scens = country_failed_scens(country)
    N = len(scens)
    slot_w = (RAIL_X1 - RAIL_X0) / max(N, 1)
    off = 0.16 * slot_w
    for si, scen in enumerate(scens):
        col = SCENARIO_COLOR.get(scen, '#444')
        xc = RAIL_X0 + (si + 0.5) * slot_w
        goal = goal_level(country, scen)
        per_model = feasibles[country][scen]          # {model: {ordering: val}}
        # collapse ascending + descending to their mean (one marker per model)
        model_mean = {mdl: float(np.mean(list(by_ord.values())))
                      for mdl, by_ord in per_model.items() if by_ord}
        all_feas = list(model_mean.values())
        if not all_feas:
            continue
        # thin connector: goal gridline level to lowest feasible
        if goal is not None:
            ax.plot([xc, xc], [goal, min(all_feas)],
                     color=col, lw=1.1, alpha=0.8, zorder=4)
            # small goal cap (short horizontal dash on the gridline)
            ax.plot([xc - off, xc + off], [goal, goal],
                     color=col, lw=1.6, alpha=0.9, zorder=5)
        # feasible markers: shape=model, mean of asc/desc orderings
        for mdl, val in model_mean.items():
            ax.scatter(xc, val,
                        s=20, marker=FEAS_MARKER[mdl],
                        facecolors=col, edgecolors='white',
                        linewidth=0.5, zorder=6)

    # countries that pass screening in every scenario have no failed
    # cases and therefore no reconstruction marker. Label them so the
    # empty strip reads as 'no reduction needed', not missing data.
    if not scens:
        ax.text((RAIL_X0 + RAIL_X1) / 2, 0.5,
                '✓ passes screening\nin all scenarios\n(no reconstruction needed)',
                transform=ax.get_xaxis_transform(), ha='center', va='center',
                fontsize=8.0, color='#117733', fontweight='bold', zorder=6)

    # cosmetics
    ax.set_yscale('log')
    ax.set_xlim(0, XLIM)
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    ax.set_box_aspect(BOX_ASPECT)
    ax.grid(axis='y', which='major', color='#eeeeee', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_title(f'({panel_tags[idx]}) {country}', fontsize=12,
                  fontweight='bold', loc='left')
    ax.annotate('goal (cap) → reconstructed (markers)',
                 xy=((RAIL_X0 + RAIL_X1) / 2, 1.0),
                 xycoords=('data', 'axes fraction'), ha='center', va='bottom',
                 fontsize=7.2, color='#666')
    if idx % ncols == 0:
        ax.set_ylabel('Required resource C (Gt)')
    if idx >= n - ncols:
        ax.set_xlabel('Growth rate (%)')

for j in range(n, len(axes)):
    axes[j].axis('off')

# Legend
samp_h = [
    Line2D([0],[0], marker='o', linestyle='', markerfacecolor=MODEL_COLOR['Logistic'],
            markeredgecolor='none', markersize=8, label='Logistic MC samples'),
    Line2D([0],[0], marker='^', linestyle='', markerfacecolor=MODEL_COLOR['Gompertz'],
            markeredgecolor='none', markersize=8, label='Gompertz MC samples'),
]
enc_h = [
    Line2D([0],[0], color='#b5b5b5', linestyle=(0,(5,3)), lw=1.0,
            label='Goal: scenario resource C (gridline + coloured cap)'),
    Line2D([0],[0], marker='o', linestyle='', markerfacecolor='#555',
            markeredgecolor='white', markeredgewidth=0.5, markersize=7,
            label='Reconstruction C — Logistic (●)'),
    Line2D([0],[0], marker='^', linestyle='', markerfacecolor='#555',
            markeredgecolor='white', markeredgewidth=0.5, markersize=7,
            label='Reconstruction C — Gompertz (▲)'),
    Line2D([0],[0], color='#888', lw=1.1,
            label='Connector length = geological shortfall'),
]
scen_h = [Line2D([0],[0], color=SCENARIO_COLOR[s], lw=3.0,
                  label=SCENARIO_LABEL.get(s, s)) for s in failed_scenarios]

leg1 = fig.legend(handles=samp_h + enc_h, loc='lower left',
                   bbox_to_anchor=(0.05, -0.02), ncol=1, frameon=False,
                   fontsize=8.3, title='Samples & dumbbell encoding',
                   title_fontsize=9, alignment='left')
fig.add_artist(leg1)
fig.legend(handles=scen_h, loc='lower right',
            bbox_to_anchor=(0.99, -0.02), ncol=2, frameon=False,
            fontsize=8.3, title='Scenario (goal cap & reconstruction marker colour)',
            title_fontsize=9, alignment='left')

fig.suptitle('Modelled geological feasibility vs set scenario goals, by country',
              fontsize=15, fontweight='bold', y=0.975)
fig.text(0.5, 0.94,
    'Each panel: MC sample cloud (growth rate vs required resource, colour = model). '
    'Right strip (same y-axis): per failed scenario, screening-informed '
    'reconstruction C markers '
    '(shape = model; mean of ascending & descending ordering) hang below the '
    'scenario goal C — the min/ref/max gridlines, marked with a coloured cap. '
    'Connector length = geological shortfall.',
    ha='center', va='top', fontsize=8.5, color='#444')

plt.subplots_adjust(top=0.90, bottom=0.16, left=0.06, right=0.99,
                     hspace=0.36, wspace=0.26)

png = FIG / 'fig_global_country_growth_vs_resource.png'
pdf = FIG / 'fig_global_country_growth_vs_resource.pdf'
FIG.mkdir(parents=True, exist_ok=True)
fig.savefig(png, dpi=200, bbox_inches='tight')
fig.savefig(pdf, dpi=240, bbox_inches='tight')
plt.close(fig)
print(f'Saved {png}')
print(f'Saved {pdf}')